In [1]:
import numpy as np
import pandas as pd

#IMPORT DATASET
movies = pd.read_csv("../dataset/tmdb_5000_movies.csv")
credits = pd.read_csv("../dataset/tmdb_5000_credits.csv")

# movies.head()
# credits.head()

# MERGE THE DATASET
movies = movies.merge(credits,on='title')
# movies.head(1)
# movies.info()

#REMOVE UNNECESSARY COLUMNS
#KEEP THESE
#COLUMNS = [genres,id,keywords,title,overview,cast,crew]

movies = movies[['movie_id','title','overview','genres','keywords','cast','crew']]
# movies.head()

#CHECK FOR NULL VALUES and delete
# movies.isnull().sum()
movies.dropna(inplace=True) #we used inplace is true because it Modify=ies the original dataframe directly instead of creating a new one

#CHECK FOR DUPLICATE VALUES
# movies.duplicated().sum()  #no duplicate rows

#extract the keywords from genres 
# print(movies.iloc[0].genre)

import ast 
def convert(obj):
    L = []
    for i in ast.literal_eval(obj): #we used ast.literal_eval to convert the indeces from string to integer
        L.append(i['name'])
    return L

movies['genres']= movies['genres'].apply(convert)
# convert('[{"id": 28, "name": "Action"}, {"id": 12, "name": "Adventure"}, {"id": 14, "name": "Fantasy"}, {"id": 878, "name": "Science Fiction"}]')
movies['keywords'] = movies['keywords'].apply(convert)


def extractCastName(obj):
    L = []
    counter=0
    for i in ast.literal_eval(obj):
        if counter != 3:
            L.append(i['name'])
            counter+=1
        else:
            break
    return L
movies['cast'] = movies['cast'].apply(extractCastName)

def extractDirectorName(obj):
    L = []
    for i in ast.literal_eval(obj):
        if i['job']=='Director':
            L.append(i['name'])
            break
    return L
movies['crew'] = movies['crew'].apply(extractDirectorName)

movies['overview'] = movies['overview'].apply(lambda x:x.split())


#WE COMBINE THE WORDS AND  REMOVE THE SPACES SO THAT MODEL DOES NOT CONFUSE
movies['genres'] = movies['genres'].apply(lambda x:[i.replace(" ","") for i in x])
movies['keywords'] = movies['keywords'].apply(lambda x:[i.replace(" ","") for i in x])
movies['cast'] = movies['cast'].apply(lambda x:[i.replace(" ","") for i in x])
movies['crew'] = movies['crew'].apply(lambda x:[i.replace(" ","") for i in x])
# movies.head()


#CONCAT THE OVERVIEW GENRES KEYWORDS CAST CREW
movies['tags'] = movies['overview']+movies['genres']+movies['keywords']+movies['cast']+movies['crew']
# movies.head()



CREATE A NEW DATAFRAME WHICH CONTAINS TAGS , MOVIE ID, TITLE

In [2]:
new_df = movies[['movie_id','title','tags']]
# new_df.head()

new_df['tags'] = new_df['tags'].apply(lambda x:" ".join(x))

#CONVERT EVERY TAG TO LOWERCASE
new_df['tags'] = new_df['tags'].apply(lambda x:x.lower())
# new_df.head()

#WE DEAL WITH SAME MEANING WORD SUCH AS LOVING AND LOVE HAVE SAME MEANING
#STEMMING

from nltk.stem.porter import PorterStemmer
ps = PorterStemmer()

def stem(text):
    y=[]
    for i in text.split():
        y.append(ps.stem(i))        #ps.stem('loving') => 'love'
    return " ".join(y)

new_df['tags'] = new_df['tags'].apply(stem)

C:\Users\ashis\AppData\Local\Temp\ipykernel_16852\2136443435.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df['tags'] = new_df['tags'].apply(lambda x:" ".join(x))
C:\Users\ashis\AppData\Local\Temp\ipykernel_16852\2136443435.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df['tags'] = new_df['tags'].apply(lambda x:x.lower())
C:\Users\ashis\AppData\Local\Temp\ipykernel_16852\2136443435.py:22: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try usin

VECTORIZATION

In [3]:
from sklearn.feature_extraction.text import CountVectorizer
cv = CountVectorizer(max_features=1000,stop_words='english')

vectors = cv.fit_transform(new_df['tags']).toarray()
# vectors
# cv.get_feature_names_out()

#FIND THE COSINE DISTANCE BETWEEN EACH MOVIE

from sklearn.metrics.pairwise import cosine_similarity
similarity = cosine_similarity(vectors)


#IF U GET A MOVIE RETURN 5 SIMILAR MOVIE TO THAT
def recommend(movie):
    movie_index = new_df[new_df['title']==movie].index[0]
    distances = similarity[movie_index]
    #we used enumerate to have a index ahead of the distance so that when we sort the index is not losed
    movies_list = sorted(list(enumerate(distances)),reverse=True,key=lambda x:x[1])[1:6]
    for i in movies_list:
        print(new_df.iloc[i[0]].title)
    

recommend('Batman Begins')



The Dark Knight
The Yards
10th & Wolf
Synecdoche, New York
Rockaway


STORE THE TRAINED OUTPUT

In [4]:
import pickle
import numpy as np

# convert only if numpy array
if isinstance(similarity, np.ndarray):
    similarity = similarity.tolist()

pickle.dump(similarity, open("../model/similarity.pkl", "wb"))
pickle.dump(new_df, open("../model/movie_list.pkl", "wb"))